# Reconocimiento de acordes en guitarra/bajo con vision por computadora

La propuesta consiste en analizar un video de una persona tocando guitarra o bajo para estimar que acorde o nota esta ejecutando. Para lograrlo, el sistema busca combinar la posicion de la mano con la ubicacion del diapason, las cuerdas y los trastes.

El objetivo inicial de este notebook no es tener ya el modelo final entrenado, sino dejar ordenada la estructura del proyecto: como se leeria el video, donde entran MediaPipe y YOLO, como se representarian los acordes y como se podria generar una tablatura basica cuando ya existan videos de referencia.

## Base tecnica de video, deteccion y seguimiento

Como se trabaja en la presentacion de Semana 16, un video puede verse como una secuencia de frames donde se agrega la dimension temporal. Por eso no basta con analizar una sola imagen: el sistema debe revisar cada frame y, al mismo tiempo, mantener cierta continuidad entre frames para que la salida no cambie de forma brusca por una deteccion aislada.

La misma presentacion separa dos ideas que aqui son importantes: deteccion y tracking. La deteccion, por ejemplo con YOLO, localiza objetos dentro de un frame; el tracking conecta esas detecciones a traves del tiempo. En este proyecto esa diferencia se usa de forma practica: YOLO serviria para ubicar partes del instrumento en cada frame, mientras que el seguimiento temporal ayudaria a estabilizar la lectura del acorde o nota que se esta tocando.

Referencia: `Semana 16 - Video y Tracking.pdf`, diapositivas 2, 10, 23 y 24.

## Arquitectura inicial

1. Entrada: video de guitarra o bajo en afinacion estandar.
2. Deteccion de mano: MediaPipe Hands obtiene landmarks de dedos.
3. Deteccion del instrumento: YOLO/Ultralytics detecta mastil, diapason, cuerdas o trastes.
4. Geometria: las puntas de los dedos se proyectan sobre el area del diapason.
5. Matching musical: el vector de trastes observado se compara contra acordes/notas predefinidas.
6. Seguimiento temporal: se filtran cambios demasiado cortos o inestables entre frames.
7. Salida: acorde/nota por momento del video, video anotado y tablatura.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

PROJECT_ROOT

In [ ]:
from chordvision.config import load_config
from chordvision.chord_matching import load_chord_shapes, match_chord
from chordvision.instrument import frets_to_notes, get_tuning
from chordvision.tablature import TabEvent, render_ascii_tab

config = load_config(PROJECT_ROOT / "configs" / "default.yaml")
shapes = load_chord_shapes(PROJECT_ROOT / config.chord_dataset, instrument=config.instrument.instrument_type)

config, len(shapes), shapes[:2]

## Dataset minimo de acordes/notas

El archivo `data/chord_shapes.example.json` guarda cada shape como un vector de trastes. La convencion es:

- las cuerdas van de la mas grave a la mas aguda;
- `-1` significa cuerda muteada o no usada;
- `0` significa cuerda al aire;
- un numero positivo representa el traste.

Para guitarra estandar se usa `E2 A2 D3 G3 B3 E4`. Para bajo estandar se usa `E1 A1 D2 G2`.

In [ ]:
observed_em = [0, 2, 2, 0, 0, 0]
print("Notas observadas:", frets_to_notes("guitar", observed_em))

match = match_chord(observed_em, shapes)
print("Match:", match)

## MediaPipe Hands

MediaPipe Hands se propone porque entrega 21 puntos de referencia por mano, incluyendo muneca, nudillos y puntas de los dedos. Para este proyecto interesan principalmente las puntas de los dedos, ya que esos puntos son los que se pueden comparar contra la posicion de las cuerdas y los trastes.

En una primera version, MediaPipe no tiene que reconocer el acorde directamente. Su trabajo seria decirnos donde estan los dedos. Luego, otra parte del sistema convierte esas posiciones visuales en informacion musical.

In [ ]:
# Ejemplo para cuando ya exista un frame real:
#
# from chordvision.hand_tracking import HandTracker
# import cv2
#
# tracker = HandTracker(max_num_hands=1)
# frame = cv2.imread(str(PROJECT_ROOT / "data" / "raw" / "frame_prueba.jpg"))
# hands = tracker.detect(frame)
# print(hands[0].fingertip_points() if hands else "No se detecto mano")

## YOLO / Ultralytics

YOLO se usaria para localizar visualmente las partes importantes del instrumento. Esto encaja con lo visto en Semana 16: YOLO es una herramienta de deteccion por frame, es decir, puede encontrar objetos o regiones en una imagen, pero no mantiene por si solo una memoria temporal del movimiento.

Por eso, en esta propuesta YOLO no se usa para adivinar el acorde directamente. Su papel seria detectar referencias visuales como mastil, diapason, cuerdas o trastes. Con esas referencias, el sistema puede construir una relacion espacial entre la mano detectada por MediaPipe y la estructura real del instrumento.

Referencia: `Semana 16 - Video y Tracking.pdf`, diapositivas 10 y 23.

In [ ]:
# Ejemplo para cuando ya existan pesos entrenados:
#
# from chordvision.instrument_detection import InstrumentDetector
# detector = InstrumentDetector(PROJECT_ROOT / config.models.yolo_weights, confidence=config.models.yolo_confidence)
# detections = detector.detect(frame)
# for det in detections:
#     print(det.label, det.confidence, det.xyxy)

## Relacion dedo-traste

La parte central del prototipo sera convertir puntos de dedos en posiciones musicales. Una primera version puede asumir una camara relativamente fija:

- YOLO detecta el area del diapason.
- Se estiman lineas o centros de cuerdas.
- Se estiman fronteras de trastes.
- Cada punta de dedo se asigna a la cuerda mas cercana y al intervalo de traste correspondiente.

Si el angulo de la camara cambia mucho, se puede mejorar esta etapa usando una calibracion del diapason o una transformacion de perspectiva. Esa mejora seria importante porque el mismo dedo puede verse en posiciones diferentes segun la inclinacion de la camara, aunque musicalmente este presionando el mismo traste.

In [ ]:
from chordvision.geometry import nearest_string_index, fret_from_x

# Datos sinteticos: posiciones de cuerdas y trastes en pixeles.
string_y_positions = [120, 150, 180, 210, 240, 270]  # grave -> aguda
fret_x_positions = [80, 130, 178, 224, 268, 310]

finger_tip_px = (190, 151)
string_idx = nearest_string_index(finger_tip_px, string_y_positions)
fret_idx = fret_from_x(finger_tip_px[0], fret_x_positions)

print("Cuerda detectada:", string_idx)
print("Traste detectado:", fret_idx)

## Pipeline sobre video

La funcion siguiente deja marcada la estructura esperada para procesar un video completo. La idea sigue la logica de Semana 16: trabajar frame por frame, detectar elementos visuales y despues consolidar la informacion temporalmente.

Todavia no se ejecuta porque faltan los videos de referencia, el modelo YOLO entrenado y una calibracion real del diapason. Aun asi, deja claro donde se conectarian las piezas del sistema.

In [ ]:
def process_video_placeholder(video_path: Path):
    """Estructura esperada para la version con video real."""
    # import cv2
    # cap = cv2.VideoCapture(str(video_path))
    # fps = cap.get(cv2.CAP_PROP_FPS) or 30
    # frame_index = 0
    # events = []
    #
    # while True:
    #     ok, frame = cap.read()
    #     if not ok:
    #         break
    #
    #     result = pipeline.process_frame(frame, frame_index=frame_index, fps=fps)
    #     if result.chord:
    #         events.append(TabEvent(result.time_s, result.chord.label, result.chord.observed_frets))
    #
    #     frame_index += 1
    #
    # cap.release()
    # return events
    raise NotImplementedError("Conectar cuando ya existan video, YOLO y calibracion.")

## Metricas propuestas

La evaluacion deberia probar el sistema en condiciones distintas, porque tocar un instrumento no siempre ocurre con la misma luz, angulo o velocidad de movimiento. Algunas metricas utiles serian:

- accuracy de acorde/nota por frame;
- accuracy por intervalo estable de tiempo;
- precision de cuerda detectada;
- error absoluto de traste;
- porcentaje de frames sin deteccion;
- errores de tablatura contra una tablatura manual de referencia.

Esto permitiria identificar si el sistema falla mas por la mano, por la deteccion del instrumento, por el angulo de la camara o por cambios rapidos entre acordes.

In [ ]:
import pandas as pd

metrics_template = pd.DataFrame([
    {"condicion": "buena_luz", "frames": 0, "accuracy_acorde": None, "error_medio_traste": None},
    {"condicion": "baja_luz", "frames": 0, "accuracy_acorde": None, "error_medio_traste": None},
    {"condicion": "angulo_lateral", "frames": 0, "accuracy_acorde": None, "error_medio_traste": None},
    {"condicion": "movimiento_rapido", "frames": 0, "accuracy_acorde": None, "error_medio_traste": None},
    {"condicion": "oclusion_parcial", "frames": 0, "accuracy_acorde": None, "error_medio_traste": None},
])
metrics_template

## Generacion inicial de tablatura

Cuando el sistema ya produzca una secuencia de eventos `(tiempo, acorde, vector de trastes)`, se puede generar una tablatura basica. Luego se puede mejorar agregando duraciones, silencios, compases y limpieza de repeticiones.

In [ ]:
events = [
    TabEvent(time_s=0.0, label="Em", frets=(0, 2, 2, 0, 0, 0)),
    TabEvent(time_s=1.2, label="G", frets=(3, 2, 0, 0, 0, 3)),
    TabEvent(time_s=2.4, label="C", frets=(-1, 3, 2, 0, 1, 0)),
]

print(render_ascii_tab(events, "guitar"))

## Registro de apoyo externo

Para mantener trazabilidad del desarrollo, se puede guardar una tabla sencilla con cualquier apoyo externo usado durante la implementacion, incluyendo IA generativa si se usa para depurar, estructurar codigo o revisar ideas.

| Fecha | Herramienta | Prompt o consulta usada | Resultado util | Como se verifico |
|---|---|---|---|---|
| Pendiente | Pendiente | Pendiente | Pendiente | Pendiente |
